# Week 4 — Pandas Introduction: DataFrames — Loading, Inspecting & Selecting
## Phase 2a Python | PORA Academy Cohort 7

By the end of this session, you will be able to:
- Import pandas and load a CSV into a DataFrame
- Use `.shape`, `.columns`, `.dtypes`, `.head()`, `.tail()`, `.info()`, `.describe()`
- Select columns and filter rows
- Use `.value_counts()` and `.nunique()`

---
*Session timing: 2 hours | AI assistance: DeepSeek (introduced today)*

## Why This Session Matters

Olist is a Brazilian e-commerce marketplace with **99,441 orders** spanning dozens of sellers, product categories, and delivery states. Every time you want to answer a business question — "How many orders are still in transit?", "Which sellers generate the most revenue?", "Where are deliveries failing?" — you need to be able to *load* the data and *interrogate* it quickly.

Until now, we have been working with raw Python lists, loops, and dictionaries. Today we meet **pandas** — the tool that turns a CSV file into a living, queryable table in seconds. A business analyst who can load and explore a 100,000-row dataset in three lines of code is immediately more valuable than one who loops through each row manually. That is what you will be able to do by the end of this session.

The Olist orders dataset alone has 8 columns and nearly 100,000 rows. With pandas, you will learn to see the *shape* of that data in an instant, spot missing values with a single command, and filter to exactly the rows you care about — all without writing a single loop.

## Setup — Connect to Your Data

Run this cell first. It mounts your Google Drive and imports pandas so the rest of the notebook can access the Olist dataset.

In [ ]:
import pandas as pd
from google.colab import drive
import os

# Mount Google Drive (you will be prompted to authorise once)
drive.mount('/content/drive')

# Path to your Olist data folder on Drive
olist_path = '/content/drive/MyDrive/olist-data'

print("✓ pandas version:", pd.__version__)
print("✓ Drive mounted — data path set to:", olist_path)

## §3 — Concept 1: Loading a CSV into a DataFrame

When you open a spreadsheet in Excel, all the rows and columns appear in a grid you can scroll, filter, and sort. `pandas` does the same thing in Python. The `pd.read_csv()` function reads a CSV file from disk and returns a **DataFrame** — a two-dimensional table with labelled columns and numbered rows.

Think of a DataFrame like a database table that lives inside your Python session. You do not need to click anything; every operation is a line of code you can reproduce, share, and automate. Once the data is in a DataFrame, a whole library of methods — `.head()`, `.info()`, `.value_counts()` — become instantly available.

The most important thing to check after loading is the **shape**: how many rows and columns does your data have? A shape of `(99441, 8)` tells you at a glance that you have nearly 100,000 orders across 8 attributes. If that number surprises you, something went wrong with the file path or encoding — shape is your first sanity check.

In [ ]:
# Load the Olist orders dataset
orders = pd.read_csv(os.path.join(olist_path, 'olist_orders_dataset.csv'))

# What Python object did we just create?
print(type(orders))
# expected: <class 'pandas.core.frame.DataFrame'>

# Rows × columns
print(orders.shape)
# expected: (99441, 8)

# Column names
print(orders.columns.tolist())
# expected: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
#            'order_approved_at', 'order_delivered_carrier_date',
#            'order_delivered_customer_date', 'order_estimated_delivery_date']

# Data types — what kind of value is in each column?
print(orders.dtypes)
# expected: all columns show dtype 'object' (string) — dates are not parsed yet

In [ ]:
# Preview the first 3 rows
orders.head(3)
# Returns a rich HTML table in Colab — shows real order data

In [ ]:
# Preview the last 3 rows
orders.tail(3)
# Useful to check that the file was not truncated on load

## §3 — Concept 2: Exploration Methods — Seeing the Full Picture

Before analysing any dataset, a data analyst always does a quick *health check*: are there missing values? What does the distribution of each column look like? pandas gives you five methods that answer these questions in seconds.

Imagine buying a used car without checking the dashboard warning lights — you would not do it. `.info()` and `.isnull().sum()` are your dashboard lights for data. They tell you immediately where values are missing, which columns are the wrong type, and whether the size of the dataset matches what you expected.

`.value_counts()` is particularly powerful for categorical columns like `order_status`. Instead of scrolling through 99,441 rows to count how many orders say "delivered", `.value_counts()` does it in one line and sorts the result from most to least frequent — exactly the order you want to read it.

In [ ]:
# .info() — the fastest way to see column types AND null counts in one view
orders.info()
# expected output:
# RangeIndex: 99441 entries, 0 to 99440
# order_approved_at           99281 non-null  (meaning 160 nulls)
# order_delivered_carrier_date  97658 non-null  (1783 nulls)
# order_delivered_customer_date 96476 non-null  (2965 nulls)

In [ ]:
# .isnull().sum() — exact null count for every column
print(orders.isnull().sum())
# expected:
# order_id                          0
# customer_id                       0
# order_status                      0
# order_purchase_timestamp          0
# order_approved_at               160
# order_delivered_carrier_date   1783
# order_delivered_customer_date  2965
# order_estimated_delivery_date     0

In [ ]:
# .value_counts() — frequency table for a categorical column
print(orders['order_status'].value_counts())
# expected:
# delivered      96478
# shipped         1107
# canceled         625
# unavailable      609
# invoiced         314
# processing       301
# created            5
# approved           2

In [ ]:
# .nunique() — how many distinct values does a column contain?
print("Unique statuses:", orders['order_status'].nunique())
# expected: 8

print("Unique order IDs:", orders['order_id'].nunique())
# expected: 99441  ← every order_id is unique

print("Unique customer IDs:", orders['customer_id'].nunique())
# expected: 99441  ← each order has a distinct customer_id

# .describe() — statistical summary (useful for numeric columns)
# For a string column it shows: count, unique, top (most common), freq
print(orders['order_status'].describe())
# expected: count 99441, unique 8, top delivered, freq 96478

## §3 — Concept 3: Selecting Columns and Filtering Rows

A DataFrame is the whole table. But most questions you ask are only about *part* of that table — a specific column, or rows that meet a condition. pandas gives you two ways to narrow your view:

**Column selection** works like looking up a dictionary key: `orders['order_status']` gives you a **Series** (a single column). Using double brackets — `orders[['order_id', 'order_status']]` — gives you a smaller **DataFrame** with only those columns. Think of it as choosing which *columns* of a spreadsheet to show.

**Row filtering** uses a *boolean mask* — a True/False value for every row. When you write `orders['order_status'] == 'delivered'`, pandas evaluates that comparison for all 99,441 rows and returns a column of True/False values. Wrapping it in `orders[...]` then keeps only the rows where the condition is True. This is the same logic as "Filter" in Excel, but written as code you can save, audit, and reuse.

In [ ]:
# Select a SINGLE column → returns a Series (one-dimensional)
status_col = orders['order_status']
print(type(status_col))
# expected: <class 'pandas.core.series.Series'>
print(status_col.head(3))

In [ ]:
# Select MULTIPLE columns → returns a DataFrame (two-dimensional)
subset = orders[['order_id', 'order_status', 'order_purchase_timestamp']]
print(type(subset))
# expected: <class 'pandas.core.frame.DataFrame'>
print(subset.shape)
# expected: (99441, 3)

In [ ]:
# Filter rows — boolean indexing
# Step 1: create the condition (produces True/False for every row)
condition = orders['order_status'] == 'delivered'
print(condition.head())
# expected: True / True / True ... (most orders are delivered)

# Step 2: apply the condition to keep only matching rows
delivered = orders[orders['order_status'] == 'delivered']
print(len(delivered))
# expected: 96478

In [ ]:
# Filter for canceled orders
canceled = orders[orders['order_status'] == 'canceled']
print(len(canceled))
# expected: 625

# Multiple conditions — use & (AND) or | (OR), with brackets around each condition
not_delivered = orders[
    (orders['order_status'] != 'delivered') &
    (orders['order_status'] != 'shipped')
]
print(len(not_delivered))
# expected: 1756  (625 canceled + 609 unavailable + 314 invoiced + 301 processing + 5 created + 2 approved)

# Combine filter + column selection in one line
canceled_ids = orders[orders['order_status'] == 'canceled']['order_id']
print(canceled_ids.head(3))

---
## 🤖 New This Week: AI Assistance with DeepSeek

From today, you have a new tool in your workflow. DeepSeek is a powerful AI assistant that can write and explain pandas code. But AI tools are only useful if you know enough to *evaluate* what they produce — which is exactly what the last three weeks have prepared you for.

**The four rules for using DeepSeek effectively:**

1. **Understand first** — Before prompting, write in plain English what you want to achieve. If you cannot describe it, you are not ready to prompt.
2. **Prompt with context** — Tell DeepSeek the DataFrame name, the relevant column names, and the specific question. Vague prompts produce vague code.
3. **Validate the output** — Every output from DeepSeek must be verified against the expected values in this curriculum. If the numbers do not match, the code is wrong — not the curriculum.
4. **Never copy blindly** — Be able to explain every line of generated code. If you cannot, ask DeepSeek to explain it before you submit.

**Example of a good prompt:**
```
I have a pandas DataFrame called `orders` with a column `order_status` 
(string values: "delivered", "shipped", "canceled", etc.).
How do I count how many orders have each status, sorted highest to lowest?
```

Try this prompt in DeepSeek now. Compare the output to what you learned in Concept 3.

## §4 — Going Deeper: Using `.isin()` for Multi-Value Filters

Boolean filtering with `==` works for a single value. But what if you want rows where `order_status` is *any one of* three values? You could chain conditions with `|`, but that gets verbose fast. The `.isin()` method is the clean solution — it reads like natural language: "keep rows where the status is *in* this list."

This is particularly useful when analysing data quality: keeping only the *active* order statuses, or excluding a set of problematic categories in one readable line. In a real business context, `.isin()` is the difference between a readable filter that a colleague can understand at a glance and a wall of `|` conditions that nobody wants to maintain.

In [ ]:
# Filter to multiple statuses using .isin()
top_3_statuses = ['delivered', 'shipped', 'invoiced']
top_3_orders = orders[orders['order_status'].isin(top_3_statuses)]
print(top_3_orders.shape)
# expected: (97899, 8)  ← 96478 + 1107 + 314

# Equivalent using chained | conditions (harder to read):
top_3_verbose = orders[
    (orders['order_status'] == 'delivered') |
    (orders['order_status'] == 'shipped') |
    (orders['order_status'] == 'invoiced')
]
print(top_3_verbose.shape)
# expected: (97899, 8)  ← same result, but harder to extend

# .isin() also works in REVERSE with ~ (tilde = NOT)
problem_statuses = ['canceled', 'unavailable']
healthy_orders = orders[~orders['order_status'].isin(problem_statuses)]
print(len(healthy_orders))
# expected: 98207  (99441 - 625 - 609)

## §5 — Common Mistakes

These are the exact errors you will encounter in the next 30 minutes. Read them *before* you get stuck.

In [ ]:
# ── COMMON MISTAKE 1 ────────────────────────────────────────
# WRONG — single brackets when selecting multiple columns:
# subset = orders['order_id', 'order_status']
# → KeyError: ('order_id', 'order_status')

# CORRECT — double brackets create a list of column names:
subset = orders[['order_id', 'order_status']]
print(subset.shape)
# expected: (99441, 2)

# ── COMMON MISTAKE 2 ────────────────────────────────────────
# WRONG — missing parentheses around conditions when using & or |:
# result = orders[orders['order_status'] != 'delivered' & orders['order_status'] != 'shipped']
# → TypeError: ambiguous truth value of a Series

# CORRECT — each condition needs its own parentheses:
result = orders[
    (orders['order_status'] != 'delivered') &
    (orders['order_status'] != 'shipped')
]
print(len(result))
# expected: 1756

# ── COMMON MISTAKE 3 ────────────────────────────────────────
# WRONG — using = instead of == in a filter:
# delivered = orders[orders['order_status'] = 'delivered']
# → SyntaxError (single = is assignment, == is comparison)

# CORRECT:
delivered_count = len(orders[orders['order_status'] == 'delivered'])
print(delivered_count)
# expected: 96478

## §6 — Mini-Challenge ⏱ ~7 minutes

Work individually. Use only what you have learned in this session.

**Scenario:** The Olist operations team wants a quick count of orders that are *stuck* — status is either `'processing'` or `'created'` (these are orders that have not progressed to shipping).

**Tasks:**
1. Filter the `orders` DataFrame to only 'processing' and 'created' orders using `.isin()`
2. How many rows does the result have? Expected: **306** (301 + 5)
3. Print the value counts of `order_status` for just this filtered subset

Expected final output:
```
processing    301
created         5
Name: order_status, dtype: int64
```

In [ ]:
# Mini-Challenge — given variables (do not change)
stuck_statuses = ['processing', 'created']

# Task 1: Filter orders to only stuck_statuses using .isin()
# Your code here:
# stuck_orders = orders[...]

# Task 2: When done with Task 1, uncomment and run:
# print(len(stuck_orders))  # expected: 306

# Task 3: Print value_counts of order_status for the filtered subset
# Your code here:
# print(stuck_orders['order_status'].value_counts())
# expected output:
# processing    301
# created         5

## §7 — Group Exercise (30 minutes)

Work in your groups. Attempt each task yourself first (5 minutes per task), then use DeepSeek if you are stuck. **Verify every answer against the expected value below before moving on.**

Using `olist_orders_dataset.csv` (already loaded as `orders`):

In [ ]:
# ── Task 1 ──────────────────────────────────────────────────
# Verify the shape of the orders DataFrame
print(orders.shape)
# expected: (99441, 8)

# ── Task 2 ──────────────────────────────────────────────────
# How many orders have NO delivery date? (order_delivered_customer_date is null)
# Your code here:
# missing_delivery = orders['order_delivered_customer_date'].isnull().sum()
# print(missing_delivery)
# expected: 2965

# ── Task 3 ──────────────────────────────────────────────────
# Filter to orders where status is 'canceled' OR 'unavailable'. How many rows?
# Your code here:
# canceled_or_unavailable = orders[orders['order_status'].isin([...])]
# print(len(canceled_or_unavailable))
# expected: 1234  (625 + 609)

# ── Task 4 ──────────────────────────────────────────────────
# What % of all orders are 'delivered'?
# Your code here:
# delivered_pct = ...
# print(f"{delivered_pct:.2f}%")
# expected: 97.01%

# ── Task 5 ──────────────────────────────────────────────────
# Filter to only the 3 most common statuses: delivered, shipped, invoiced
# Hint: use .isin()
# Your code here:
# top_3 = orders[orders['order_status'].isin([...])]
# print(len(top_3))
# expected: 97899  (96478 + 1107 + 314)

## §8 — Session Summary

| Concept | Key Syntax | Quick Example |
|---|---|---|
| Load a CSV | `pd.read_csv('file.csv')` | `orders = pd.read_csv('olist_orders_dataset.csv')` |
| Check dimensions | `.shape` | `orders.shape` → `(99441, 8)` |
| List columns | `.columns.tolist()` | 8 column names |
| Column types + nulls | `.info()` | See all at once |
| Count nulls | `.isnull().sum()` | `2965` missing delivery dates |
| Frequency table | `.value_counts()` | `delivered: 96478` |
| Count distinct values | `.nunique()` | `8` order statuses |
| Select one column | `df['col']` → Series | `orders['order_status']` |
| Select many columns | `df[['col1', 'col2']]` → DataFrame | Shape: `(99441, 2)` |
| Filter rows | `df[df['col'] == value]` | `96478` delivered orders |
| Filter multi-value | `df[df['col'].isin([...])]` | `97899` top-3-status orders |

---

**Coming up Thursday**: We load the **order items** dataset, add calculated columns (total cost, price tier), sort by value, and find the top sellers by revenue. You will also chain multiple pandas operations together in a single expression.